In [ ]:
## main code

In [3]:
import os
import importlib
import util
importlib.reload(util)
from util import (make_out_folder, load_naps, load_compact, compact_to_15, merge_data, show_metrics, save_line,
    save_scatter, run_q1, q1_rh_bias, run_q2, run_q3,)

def main():
    input = r"E:\Comp_exam_UofT\Comp_EXam_Questions_Manisha_1_2026-04-06\NAPS Reference and Compact Station Data Bakhtari.xlsx"
    out = r"E:\Comp_exam_UofT\Writting\Manisha comp\Bakhtari\code_out\air_sensor_outputs"
    x = load_naps(input) # naps
    y = load_compact(input, x=6) #compact raw
    x1 = compact_to_15(y) # compact 15 min
    y1 = merge_data(x, x1) # merged data
    y1.to_csv(os.path.join(out, "merged_reference_compact_15min.csv"), index=False)

    # Q1
    x2 = run_q1(y1, y=True, a=False)
    y2 = q1_rh_bias(y1)
    y2.to_csv(os.path.join(out, "q1_pm25_bias_by_rh_band.csv"), index=False)
    x2.out.to_csv(os.path.join(out, "q1_pm25_test_predictions.csv"), index=False)

    print("Q1. PM2.5 humidity correction")
    show_metrics("Raw model (PM only)", x2.x)
    show_metrics("Corrected model (PM + RH + Temp)", x2.y)

    if x2.a is not None:
        show_metrics("RH > 75% | Raw model", x2.a)
        show_metrics("RH > 75% | Corrected model", x2.b)

    if x2.c is not None:
        show_metrics("RH > 85% | Raw model", x2.c)
        show_metrics("RH > 85% | Corrected model", x2.d)

    print("\nPM bias by RH band:")
    print(y2.to_string(index=False))

    save_line(
        x2.out,
        ["PM25 Ug/m3", "PM-2.5", "pred_corr"],
        out,
        "Q1 PM2.5: Reference vs Raw Compact vs Corrected Prediction",
        "q1_pm25_timeseries.png",
    )

    x3 = y1.dropna(subset=["RH_ref", "PM-2.5", "PM25 Ug/m3"]).copy()
    x3["pm_bias"] = x3["PM-2.5"] - x3["PM25 Ug/m3"]

    save_scatter(
        x3,
        "RH_ref",
        "pm_bias",
        out,
        "Q1 PM2.5 Bias vs Relative Humidity",
        "q1_pm25_bias_vs_rh.png",
    )

    # Q2
    y3 = run_q2(y, x)
    y3.z.to_csv(os.path.join(out, "q2_no2_alignment_grid_search.csv"), index=False)
    y3.out.to_csv(os.path.join(out, "q2_no2_best_aligned_series.csv"), index=False)
  
    print("Q2. NO2 temporal harmonization and lag")
    show_metrics("Baseline comparison", y3.x)
    show_metrics(f"Best alignment (window={y3.a} min, lag={y3.b} min)", y3.y)
    x4 = y3.out.rename(columns={"NO2 ppb": "NO2_ref", "NO2_smoothed": "NO2_compact_aligned"})

    save_line(
        x4,
        ["NO2_ref", "NO2_compact_aligned"],
        out,
        f"Q2 NO2 Alignment: Best window={y3.a} min, lag={y3.b} min",
        "q2_no2_aligned_timeseries.png",
    )

    # Q3
    y4 = run_q3(y1, y=False)

    y4.out.to_csv(os.path.join(out, "q3_no2_cross_sensitivity_predictions.csv"), index=False)

    print("Q3. NO2-O3 cross-sensitivity fusion")
    show_metrics("Model 1: raw NO2 only", y4.x)
    show_metrics("Model 2: raw NO2 + O3 + met", y4.y)
    show_metrics("Model 3: dynamic O3 + afternoon + met", y4.a)

    if y4.b is not None:
        show_metrics("Afternoon only | Model 1", y4.b)
        show_metrics("Afternoon only | Model 2", y4.c)
        show_metrics("Afternoon only | Model 3", y4.d)

    print("\nCross-sensitivity diagnostics:")
    for x5, y5 in y4.e.items():
        print(f"  {x5}: {y5:.3f}")

    x6 = y4.out.rename(
        columns={
            "NO2 ppb": "NO2_ref",
            "NO2_ppb": "NO2_raw_compact",
            "pred_m3": "NO2_dynamic_corrected",
        }
    )
    save_line(
        x6,
        ["NO2_ref", "NO2_raw_compact", "NO2_dynamic_corrected"],
        out,
        "Q3 NO2: Reference vs Raw Compact vs Dynamic O3-Corrected Prediction",
        "q3_no2_timeseries.png",
    )
    save_scatter(
        y4.out,
        "O3_ppb",
        "NO2_ppb",
        out,
        "Q3 Raw compact NO2 vs compact O3",
        "q3_raw_no2_vs_o3.png",
    )
main()